In [ ]:
# Usuário, senha, host e port do banco de dados
user = dbutils.secrets.get("databases_lh_nautical", "sql_server_user")
password = dbutils.secrets.get("databases_lh_nautical", "sql_server_password")
host = dbutils.secrets.get("databases_lh_nautical", "sql_server_host")
port = dbutils.secrets.get("databases_lh_nautical", "sql_server_port")

In [ ]:
# Info do banco de dados
jdbc_url = f"jdbc:sqlserver://{host}:{port};databaseName=lh_nautical;encrypt=false;trustServerCertificate=true;"
connection_properties = {
    "user": user,
    "password": password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

# Info do catalog do desenvolvedor
catalog_dev = dbutils.widgets.get("catalog_dev")
schema = dbutils.widgets.get("schema")

In [ ]:
# Query para descobrir todas as tabelas do esquema 'dbo'
query_tabelas = "(select table_name from information_schema.tables where table_schema = 'dbo' and table_type = 'Base table') as tabelas"

# Puxamos as lista de tabelas para um DataFrame e convertemos para uma lista em Python
df_tabelas = spark.read.jdbc(
  url = jdbc_url, 
  table = query_tabelas, 
  properties = connection_properties
)

tabelas = df_tabelas.collect() 
lista_tabelas = [row['table_name'] for row in df_tabelas.collect()]

In [ ]:
# Ingestão das tabelas no DW
for nome_tabela in lista_tabelas:
    print(f"Importanto a tabela: {nome_tabela}...")
    tabela_completa = f"dbo.{nome_tabela}"


    if nome_tabela not in ['variant_attribute_values', 'product_suppliers', 'stock_levels']:

        bounds_df = spark.read.jdbc(
            url = jdbc_url,
            table = f"(SELECT MIN(id) AS min_id, MAX(id) AS max_id FROM {tabela_completa}) AS bounds",
            properties = connection_properties
        )

        bounds = bounds_df.collect()[0]
        lower_id = bounds['min_id']
        upper_id = bounds['max_id']


        df_temp = spark.read.jdbc(
            url = jdbc_url, 
            table = tabela_completa,
            column = "id",
            lowerBound = lower_id,
            upperBound = upper_id,
            numPartitions = 8, 
            properties = connection_properties
        )
    elif nome_tabela in ['product_suppliers', 'stock_levels']:
        
        bounds_df = spark.read.jdbc(
            url = jdbc_url,
            table = f"(SELECT MIN(updated_at) AS min_date, MAX(updated_at) AS max_date FROM {tabela_completa}) AS bound",
            properties = connection_properties
        )

        bounds = bounds_df.collect()[0]
        lower_date = bounds['min_date']
        upper_date = bounds['max_date']

        df_temp = spark.read.jdbc(
        url = jdbc_url, 
        table = tabela_completa,
        column = "updated_at",
        lowerBound = lower_date,
        upperBound = upper_date,
        numPartitions = 8, 
        properties = connection_properties
    )

    else:
        df_temp = spark.read.jdbc(
            url = jdbc_url, 
            table = tabela_completa,
            properties = connection_properties
        )
  
    df_temp.write.format("delta").mode("append").saveAsTable(f"{catalog_dev}.{schema}.{nome_tabela}")

print("Ingestão finalizada!!")